# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step example for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset information
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Let's inspect the dataset schema:

- List available record sets
- List their fields and the corresponding `@id` for reference

Below we print the record set `@id`s and details.

In [ ]:
# List record sets (`@id` and name)
record_set_ids = []
print("Available record sets:")
for record_set in dataset.metadata.record_sets:
    print(f"- @id: {record_set.id}, Name: {record_set.name}")
    record_set_ids.append(record_set.id)
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - @id: {field.id}, Name: {field.name}, Data type: {field.data_type}")

# For demonstration, pick the first available record set
if len(record_set_ids) > 0:
    chosen_record_set_id = record_set_ids[0]
    print(f"\nExample record set for extraction: {chosen_record_set_id}")
else:
    raise ValueError("No record sets found in dataset metadata.")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. Always reference record set and field IDs by their `@id`.

Here we extract all records for the main (first) record set.

In [ ]:
dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
        else:
            df = pd.DataFrame()
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for record set: {rs_id} with shape {df.shape}")
    except Exception as e:
        print(f"Failed loading records for {rs_id}: {e}")

# Show example columns and head for the chosen record set
print(f"\nColumns in DataFrame for record set {chosen_record_set_id}:")
print(dataframes[chosen_record_set_id].columns.tolist())
dataframes[chosen_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps using field `@id` as a reference. Typical EDA includes filtering for specific numeric values, normalization, or categorization. 

**Note:** If there are no obvious numeric fields, update the variables below with the actual numeric field `@id` (as printed above).

In [ ]:
# Choose a numeric field (`@id`)
# Replace below with correct field `@id` (column name) as printed in overview. For demonstration, we'll do it dynamically if possible.

numeric_field = None
possible_numeric_types = {'Number', 'Float', 'Integer'}
for record_set in dataset.metadata.record_sets:
    if record_set.id == chosen_record_set_id:
        for field in record_set.fields:
            if field.data_type in possible_numeric_types:
                numeric_field = field.id
                print(f"Selected numeric field for EDA: {numeric_field}")
                break
        break
if numeric_field is None:
    raise ValueError("No numeric field detected in the chosen record set. Please update the field id variable explicitly.")

numeric_col_name = numeric_field  # Field @id is used as DataFrame column

# Show some stats
print(f"Summary statistics for field {numeric_col_name}:")
print(dataframes[chosen_record_set_id][numeric_col_name].describe())

# Filtering
threshold = dataframes[chosen_record_set_id][numeric_col_name].mean()
filtered_df = dataframes[chosen_record_set_id][dataframes[chosen_record_set_id][numeric_col_name] > threshold]
print(f"\nFiltered records with {numeric_col_name} > {threshold:.3f}:")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_col_name}_normalized"] = (
    filtered_df[numeric_col_name] - filtered_df[numeric_col_name].mean()
) / filtered_df[numeric_col_name].std()
print(f"\nFirst 5 normalized values for {numeric_col_name}:")
print(filtered_df[[numeric_col_name, f"{numeric_col_name}_normalized"]].head())

# Try grouping by a categorical field
group_field = None
for record_set in dataset.metadata.record_sets:
    if record_set.id == chosen_record_set_id:
        for field in record_set.fields:
            if field.data_type == 'Text' and field.id != numeric_field:
                group_field = field.id
                break
        break
if group_field and group_field in dataframes[chosen_record_set_id].columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_col_name].mean().reset_index()
    print(f"\nGrouped by field {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and the effect of filtering/normalizing. Feel free to modify the field `@id`s for other columns relevant to your analysis.

In [ ]:
import matplotlib.pyplot as plt

# Plot distribution of the numeric field
plt.figure(figsize=(8,4))
dataframes[chosen_record_set_id][numeric_col_name].hist(bins=40)
plt.title(f"Distribution of {numeric_col_name}")
plt.xlabel(numeric_col_name)
plt.ylabel("Frequency")
plt.show()

# Show the effect of normalization
plt.figure(figsize=(8,4))
if f"{numeric_col_name}_normalized" in filtered_df.columns:
    filtered_df[f"{numeric_col_name}_normalized"].hist(bins=40)
    plt.title(f"Normalized {numeric_col_name} (filtered)")
    plt.xlabel(f"Normalized {numeric_col_name}")
    plt.ylabel("Frequency")
    plt.show()
else:
    print("No normalized column to plot.")

## 6. Conclusion
- We loaded FAIR^2 mass spectrometry records using the Croissant schema and `mlcroissant` package.
- We explored the record set and fields using their `@id` identifiers.
- We demonstrated basic EDA and plotted the distribution of a primary numeric variable.

**Next steps** might include outlier removal, advanced domain-specific groupings, or downstream analysis such as machine learning with selected features.